# ShorlyNot — Teleportation-Assisted Quantum Digital Signatures & Statistical Threat Detection
**Team Momosapiens (`VTU-QFF-HC-0085`)** | **VTU-Qiskit Fall Fest 2026**

This notebook explores the core quantum mechanics and statistical guarantees behind **ShorlyNot**:
1. **3-Qubit Bell Teleportation Circuit (`ShorlyNot-QDS-T1`)** on Qiskit `AerSimulator` with dynamic feed-forward (`if_test`).
2. **Honest vs. Unentangled Forger Verification** — comparing exact $O(n)$ Born-rule projection against full `AerSimulator` circuit execution.
3. **Attack-Strength ROC Analysis** — Hoeffding concentration bounds vs. exact binomial thresholds across partial-information adversaries.
4. **Attribution-Aware Response Policy** — verifying that `ML-DSA-44` identity signatures prevent false-flag victim lockouts.

In [ ]:
import hashlib
import matplotlib.pyplot as plt
from shorlynot_skeleton.engines.qiskit_aer import QiskitAerEngine
from shorlynot_skeleton.qds.protocol import QdsT1Protocol
from shorlynot_skeleton.attacks.forgery import ForgeryAttack
from shorlynot_skeleton.analysis.roc import sweep, tolerated_information, validate_against_engine

## 1. Inspecting the 3-Qubit Qiskit Teleportation Sign Circuit
Each check position encodes a payload digest bit in either the Pauli-$Z$ ($\{|0\rangle, |1\rangle\}$) or Pauli-$X$ ($\{|+\rangle, |-\rangle\}$) eigenbasis on `q[0]`, shares an EPR pair $|\Phi^+\rangle$ across `(q[1], q[2])`, measures `(q[0], q[1])` in the Bell basis to obtain syndrome $(m_1, m_2)$, and applies classically conditioned $X^{m_2} Z^{m_1}$ recovery on `q[2]`.

In [ ]:
qc = QiskitAerEngine.build_teleportation_circuit(bit=1, basis=1)
print(qc.draw(output='text'))

## 2. Signing & Verifying Honest vs. Forged Bundles (`analytic` & `circuit` Modes)

In [ ]:
protocol = QdsT1Protocol(p0=0.02, default_delta=0.01)
payload_hash = hashlib.sha256(b'alice->bob:INR:2500').hexdigest()

honest_bundle = protocol.sign(payload_hash=payload_hash, key_id='alice-key-1', n_checks=64, L=128)
forged_bundle = ForgeryAttack.generate_forged_bundle(payload_hash=payload_hash, claimed_key_id='alice-key-1')

for label, bundle in [('Honest', honest_bundle), ('Forged', forged_bundle)]:
    r_analytic = protocol.verify(bundle, mode='analytic')
    r_circuit = protocol.verify(bundle, mode='circuit')
    print(f'{label:<7} | Analytic: accepted={str(r_analytic.candidate_accepted):<5} p_hat={r_analytic.mismatch_rate:.4f} '
          f'| Circuit: accepted={str(r_circuit.candidate_accepted):<5} p_hat={r_circuit.mismatch_rate:.4f} (tau={r_analytic.tau:.4f})')

## 3. Partial-Information Forger ROC Curve (Hoeffding vs. Exact Binomial Threshold)

In [ ]:
roc = sweep(n=64, p0=0.02, delta=0.01)
bounds = tolerated_information(n=64, p0=0.02, delta=0.01)
mc_check = validate_against_engine(n=64, p0=0.02, trials=200)
print('Attacker syndrome knowledge required for 50% acceptance:', bounds)
print('Theory vs Engine Monte Carlo check:', mc_check)

plt.figure(figsize=(7, 4))
plt.plot(roc['x'], roc['partial_info_hoeffding'], label=f"Hoeffding (tau={roc['tau']['hoeffding']:.3f})", lw=2)
plt.plot(roc['x'], roc['partial_info_exact'], label=f"Exact Binomial (tau={roc['tau']['exact']:.3f})", lw=2, linestyle='--')
plt.xlabel('Fraction f of True Syndromes Known to Attacker')
plt.ylabel('Acceptance Probability P(accept)')
plt.title('Q-STDF Detector Resilience Against Partial-Information Forgers (n=64)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. End-to-End Pipeline & Attribution-Scoped Anti-Lockout Verification

In [ ]:
from shorlynot_skeleton.pipeline import QuantumTransferPipeline
from shorlynot_skeleton.models import TransactionPayload, TransferPipelineRequest

pipeline = QuantumTransferPipeline()

# Attacker injects forged bundle claiming to be 'alice'
attack_res = pipeline.execute_transfer(TransferPipelineRequest(
    transaction=TransactionPayload(from_user='alice', to_user='bob', amount=9000.0, tx_id='tx-atk-01'),
    signer_key_id='alice-key-1',
    verifier_id='bob',
    simulate_attack='forgery',
    source_id='attacker-node-99'
))

# Legitimate Alice continues transacting from her authenticated origin without lockout
alice_res = pipeline.execute_transfer(TransferPipelineRequest(
    transaction=TransactionPayload(from_user='alice', to_user='bob', amount=500.0, tx_id='tx-honest-02'),
    signer_key_id='alice-key-1',
    verifier_id='bob',
    source_id='alice-banking-session'
))

print('Forged attempt success :', attack_res.success, '| Stage:', attack_res.stage_s.name)
print('Honest Alice transfer  :', alice_res.success, '| Message:', alice_res.message)